In [ ]:
%%configure -f
{
    "conf": {
        "spark.dynamicAllocation.enabled": "false",
        "spark.driver.cores": "4",
        "spark.driver.memory": "28g",
        "spark.executor.cores": "4",
        "spark.executor.memory": "28g",
        "spark.executor.instances": 1
    }
}

# Notebook 06: Train / Test / Select / Tune

**Scenario:** supermarket_net_sales_forecast  
**Generated:** 2026-06-03  
**Key Parameters:**
- Forecast Horizon: 4 weeks
- Time Granularity: Weekly (W-THU)
- Target Column: TOTAL_NET_SALES
- Series ID: STORE_LOCATION_ID
- Model: LightGBM (via MLForecast/Nixtla)
- Lags: [4, 8, 13, 26]
- Best Transform: LocalStandardScaler (std)
- Evaluation: Rolling-origin (8 cutoffs) + 4-week OOS

---

## Modeling Framework

All models use the **Nixtla MLForecast** ecosystem with **LightGBM** as the base learner.  
Three target transformations are compared:
1. **Identity** â€” raw target
2. **Standard Scaled** â€” LocalStandardScaler (per-series z-score)
3. **Differenced** â€” first differences to remove trend

The best transform is selected by overall MAE on rolling-origin evaluation.

## 1. Package Imports

In [ ]:
import numpy as np
import pandas as pd
import time
from mlforecast import MLForecast
from mlforecast.target_transforms import LocalStandardScaler, Differences
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

## 2. Configuration Parameters

In [ ]:
# Scenario configuration
scenario_name = "supermarket_net_sales_forecast"
INPUT_TABLE = f"{scenario_name}_features"
OUTPUT_TABLE = f"{scenario_name}_forecasts"

# Column roles
unique_id = 'STORE_LOCATION_ID'
y = 'TOTAL_NET_SALES'
date_col = 'WEEK_START_DT'

# Forecasting parameters
freq = 'W-THU'
horizon = 4
lags = [4, 8, 13, 26]

# Evaluation dates
training_start = '2024-04-25'
training_end = '2025-02-27'
forecast_start = '2025-03-06'
forecast_end = '2025-03-27'

# Model selection metric
selection_metric = 'MAE'

print(f"Scenario: {scenario_name}")
print(f"Input: {INPUT_TABLE}, Output: {OUTPUT_TABLE}")
print(f"Horizon: {horizon} weeks, Frequency: {freq}")
print(f"Lags: {lags}")
print(f"Training: {training_start} â†’ {training_end}")
print(f"OOS Forecast: {forecast_start} â†’ {forecast_end}")

## 3. Load Feature Table

In [ ]:
# Load feature table from lakehouse
df = spark.sql(f"SELECT * FROM {INPUT_TABLE}").toPandas()

# Rename date column to 'ds' for MLForecast
df = df.rename(columns={date_col: 'ds'})
df['ds'] = pd.to_datetime(df['ds'])
df = df.sort_values([unique_id, 'ds']).reset_index(drop=True)

print(f"Feature table loaded: {df.shape}")
print(f"Date range: {df['ds'].min()} to {df['ds'].max()}")
print(f"Stores: {df[unique_id].nunique()}")
print(f"Weeks per store: {df.groupby(unique_id).size().unique()}")
print(f"Clusters: {df['profile_cluster'].value_counts().to_dict()}")

## 4. Drop Pre-Computed Lag/Rolling Columns

MLForecast will regenerate lags internally. We drop the pre-computed ones from NB05 to avoid duplication.

In [ ]:
# CUSTOMIZED: Drop pre-computed lag/rolling columns â€” MLForecast regenerates these
lag_rolling_cols = [c for c in df.columns if c.startswith('lag') or c.startswith('rolling_')]
print(f"Dropping {len(lag_rolling_cols)} pre-computed lag/rolling columns: {lag_rolling_cols}")
df = df.drop(columns=lag_rolling_cols)
print(f"Shape after drop: {df.shape}")

## 5. Define Evaluation Dates & Transform Configs

In [ ]:
# Build evaluation date lists
all_dates = sorted(df['ds'].unique())

# Training evaluation dates: last 8 dates within training period
eval_date_training = [d for d in all_dates if pd.Timestamp(training_start) <= pd.Timestamp(d) <= pd.Timestamp(training_end)]
eval_dates_rolling = eval_date_training[-8:]  # 8 rolling-origin cutoffs

# OOS evaluation dates: dates after training period
eval_date_oos = [d for d in all_dates if pd.Timestamp(d) > pd.Timestamp(training_end)]

print(f"Rolling eval cutoffs ({len(eval_dates_rolling)}): {[str(d)[:10] for d in eval_dates_rolling]}")
print(f"OOS eval dates ({len(eval_date_oos)}): {[str(d)[:10] for d in eval_date_oos]}")

# Transform configurations
transform_configs = {
    'identity': [],
    'std': [LocalStandardScaler()],
    'diff1': [Differences([1])],
}

# Identify feature columns (exclude meta columns)
meta_cols = [unique_id, 'ds', y, 'profile_cluster']
feature_cols = [c for c in df.columns if c not in meta_cols]
print(f"\nFeature columns: {len(feature_cols)}")

## 6. Utility Functions

In [ ]:
class Utils:
    """Utility class for column detection and type handling."""
    
    @staticmethod
    def get_static_cols(df, id_col):
        """Identify columns that are constant within each series (static features)."""
        static = []
        for col in df.columns:
            if col in [id_col, 'ds']:
                continue
            if df.groupby(id_col)[col].nunique().max() == 1:
                static.append(col)
        return static
    
    @staticmethod
    def get_binary_cols(df, exclude_cols=None):
        """Identify columns with only 0/1 values."""
        if exclude_cols is None:
            exclude_cols = []
        binary = []
        for col in df.columns:
            if col in exclude_cols:
                continue
            vals = df[col].dropna().unique()
            if set(vals).issubset({0, 1, 0.0, 1.0}):
                binary.append(col)
        return binary
    
    @staticmethod
    def find_date(df, eval_date, direction='backward'):
        """Find nearest date in df to eval_date."""
        dates = sorted(df['ds'].unique())
        target = pd.Timestamp(eval_date)
        if direction == 'backward':
            valid = [d for d in dates if pd.Timestamp(d) <= target]
            return valid[-1] if valid else dates[0]
        else:
            valid = [d for d in dates if pd.Timestamp(d) >= target]
            return valid[0] if valid else dates[-1]
    
    @staticmethod
    def get_proper_data_types(df, id_col, y_col):
        """Ensure proper dtypes for MLForecast."""
        df = df.copy()
        df['ds'] = pd.to_datetime(df['ds'])
        df[y_col] = df[y_col].astype(float)
        return df

In [ ]:
class SimilarDay:
    """Fill exogenous features using similar-day strategy."""
    
    @staticmethod
    def fill_exog_similar_week(df, id_col, feature_cols, horizon):
        """Fill future feature values using same week from prior year."""
        df = df.copy()
        for col in feature_cols:
            mask = df[col].isna()
            if mask.any():
                df.loc[mask, col] = df.groupby(id_col)[col].shift(52).loc[mask]
        return df


class StandardDemand:
    """Fill exogenous features using standard consumption (weekly mean)."""
    
    @staticmethod
    def fill_exog_standard_consumption(df, id_col, feature_cols):
        """Fill NaN with per-series mean."""
        df = df.copy()
        for col in feature_cols:
            mask = df[col].isna()
            if mask.any():
                means = df.groupby(id_col)[col].transform('mean')
                df.loc[mask, col] = means.loc[mask]
        return df


class LastKnownValue:
    """Fill exogenous features using last known value (forward fill)."""
    
    @staticmethod
    def fill_exog_last_known(df, id_col, feature_cols):
        """Forward-fill NaN values within each series."""
        df = df.copy()
        for col in feature_cols:
            df[col] = df.groupby(id_col)[col].ffill()
        return df

## 7. Identify Static vs Dynamic Features

In [ ]:
# Identify static features (constant within each store)
static_features_list = Utils.get_static_cols(df, unique_id)
# Remove non-feature columns from static list
static_features_list = [c for c in static_features_list if c not in meta_cols]

# Dynamic features = all feature cols minus static
dynamic_cols = [c for c in feature_cols if c not in static_features_list]

print(f"Static features ({len(static_features_list)}): {static_features_list}")
print(f"Dynamic features ({len(dynamic_cols)}): {len(dynamic_cols)} columns")
print(f"  First 10: {dynamic_cols[:10]}")

## 8. Rolling-Origin Training Function

This function trains a LightGBM model using MLForecast with rolling-origin evaluation.  
**Key design:** Static features are passed via `static_features=` parameter during `fit()`,  
while `X_df` at predict time only contains dynamic (time-varying) features.

In [ ]:
def rolling_train_and_predict_with_transform_oos_v2(
    df, lags, freq, id_col, y_col,
    static_features, dynamic_features,
    eval_date_training, eval_date_oos,
    horizon, target_transforms
):
    """
    Rolling-origin train/predict with proper static/dynamic feature handling.
    
    MLForecast stores static features during fit(). At predict time,
    X_df must only contain dynamic (time-varying) features.
    
    Parameters
    ----------
    df : pd.DataFrame with columns [id_col, 'ds', y_col] + features
    lags : list of int
    freq : str (e.g., 'W-THU')
    id_col, y_col : str
    static_features : list of str (constant per series)
    dynamic_features : list of str (time-varying)
    eval_date_training : list of dates for rolling cutoffs
    eval_date_oos : list of dates for out-of-sample
    horizon : int
    target_transforms : list of transform objects
    
    Returns
    -------
    (all_preds_df, last_model) : tuple
    """
    all_preds = []
    model_out = None
    
    # All cutoff dates: rolling + OOS
    all_cutoff_dates = list(eval_date_training) + list(eval_date_oos)
    
    for cutoff_date in all_cutoff_dates:
        cutoff_ts = pd.Timestamp(cutoff_date)
        
        # Split: train = all data up to and including cutoff
        train_df = df[df['ds'] <= cutoff_ts].copy()
        
        # Future dates for prediction
        future_dates = sorted([d for d in df['ds'].unique() if pd.Timestamp(d) > cutoff_ts])[:horizon]
        if not future_dates:
            continue
        
        # All features for fitting (static + dynamic)
        all_features = static_features + dynamic_features
        fit_cols = [id_col, 'ds', y_col] + [c for c in all_features if c in train_df.columns]
        train_fit = train_df[fit_cols].copy()
        
        # Initialize MLForecast
        mlf = MLForecast(
            models=[LGBMRegressor(n_estimators=100, learning_rate=0.1, num_leaves=31,
                                  random_state=42, verbosity=-1)],
            freq=freq,
            lags=lags,
            target_transforms=target_transforms,
        )
        
        # Fit with static_features parameter
        static_in_df = [c for c in static_features if c in train_fit.columns]
        mlf.fit(train_fit, id_col=id_col, time_col='ds', target_col=y_col,
                static_features=static_in_df)
        
        # Prepare X_df for prediction â€” ONLY dynamic features
        future_df = df[df['ds'].isin(future_dates)][[id_col, 'ds'] + 
                      [c for c in dynamic_features if c in df.columns]].copy()
        
        # Fill any NaN in future features
        for col in dynamic_features:
            if col in future_df.columns and future_df[col].isna().any():
                future_df[col] = future_df[col].fillna(0)
        
        # Predict
        preds = mlf.predict(h=len(future_dates), X_df=future_df)
        
        # Merge with actuals
        actuals = df[df['ds'].isin(future_dates)][[id_col, 'ds', y_col]].copy()
        merged = preds.merge(actuals, on=[id_col, 'ds'], how='left')
        merged['cutoff_date'] = cutoff_ts
        all_preds.append(merged)
        model_out = mlf
    
    all_preds_df = pd.concat(all_preds, ignore_index=True)
    return all_preds_df, model_out

## 9. Metrics & Selection Functions

In [ ]:
def compute_all_metrics(preds_df, id_col, y_col, pred_col='LGBMRegressor'):
    """
    Compute MAE, RMSE, MAPE, SMAPE per series and overall.
    """
    valid = preds_df.dropna(subset=[y_col]).copy()
    if valid.empty:
        return pd.DataFrame()
    
    results = []
    for sid, grp in valid.groupby(id_col):
        actual = grp[y_col].values
        predicted = grp[pred_col].values
        mae = np.mean(np.abs(actual - predicted))
        rmse = np.sqrt(np.mean((actual - predicted) ** 2))
        mape = np.mean(np.abs((actual - predicted) / np.where(actual == 0, 1, actual))) * 100
        smape = np.mean(2 * np.abs(actual - predicted) / (np.abs(actual) + np.abs(predicted) + 1e-8)) * 100
        results.append({id_col: sid, 'MAE': mae, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape, 'n_preds': len(grp)})
    
    metrics_df = pd.DataFrame(results)
    
    # Add overall row
    actual_all = valid[y_col].values
    predicted_all = valid[pred_col].values
    overall = {
        id_col: '__OVERALL__',
        'MAE': np.mean(np.abs(actual_all - predicted_all)),
        'RMSE': np.sqrt(np.mean((actual_all - predicted_all) ** 2)),
        'MAPE': np.mean(np.abs((actual_all - predicted_all) / np.where(actual_all == 0, 1, actual_all))) * 100,
        'SMAPE': np.mean(2 * np.abs(actual_all - predicted_all) / (np.abs(actual_all) + np.abs(predicted_all) + 1e-8)) * 100,
        'n_preds': len(valid)
    }
    metrics_df = pd.concat([metrics_df, pd.DataFrame([overall])], ignore_index=True)
    return metrics_df


def get_target_transforms_from_name(name):
    """Get transform objects from config name."""
    configs = {
        'identity': [],
        'std': [LocalStandardScaler()],
        'diff1': [Differences([1])],
    }
    return configs[name]


def select_best_transform(scores, metric='MAE'):
    """Select transform with lowest score."""
    best_name = min(scores, key=scores.get)
    return best_name, scores[best_name]

## 10. Global Model Training â€” All Transforms

Train a single global LightGBM model across all 50 stores for each transform variant.  
Evaluation uses 8 rolling-origin cutoffs + 4-week out-of-sample.

In [ ]:
# Train global model with each transform
results_global = {}
scores = {}

for transform_name, transforms in transform_configs.items():
    print(f"\n{'='*50}")
    print(f"Training global model: {transform_name}")
    print(f"{'='*50}")
    
    t0 = time.time()
    preds, model = rolling_train_and_predict_with_transform_oos_v2(
        df=df,
        lags=lags,
        freq=freq,
        id_col=unique_id,
        y_col=y,
        static_features=static_features_list,
        dynamic_features=dynamic_cols,
        eval_date_training=eval_dates_rolling,
        eval_date_oos=eval_date_oos,
        horizon=horizon,
        target_transforms=transforms,
    )
    elapsed = time.time() - t0
    
    # Compute overall MAE on rolling evaluation (exclude OOS for selection)
    rolling_preds = preds[preds['cutoff_date'].isin([pd.Timestamp(d) for d in eval_dates_rolling])]
    valid_rolling = rolling_preds.dropna(subset=[y])
    mae = np.mean(np.abs(valid_rolling[y].values - valid_rolling['LGBMRegressor'].values))
    
    results_global[transform_name] = (preds, model)
    scores[transform_name] = mae
    print(f"  âœ… {elapsed:.1f}s | {len(valid_rolling)} eval preds | MAE: {mae:,.2f}")

print(f"\n{'='*50}")
print("TRANSFORM COMPARISON (Rolling-Origin MAE):")
for name, score in sorted(scores.items(), key=lambda x: x[1]):
    print(f"  {name:<12} MAE: {score:>12,.2f}")

## 11. Select Best Transform & Extract Metrics

In [ ]:
# Select best transform
best_transform, best_score = select_best_transform(scores, selection_metric)
best_preds, best_model = results_global[best_transform]

print(f"âœ… Best Transform: {best_transform} (MAE: {best_score:,.2f})")
print(f"   Transforms tested: {list(scores.keys())}")
print(f"   Scores: {', '.join(f'{k}={v:,.0f}' for k,v in scores.items())}")

# Compute detailed per-store metrics
global_metrics = compute_all_metrics(best_preds, unique_id, y)
overall = global_metrics[global_metrics[unique_id] == '__OVERALL__'].iloc[0]
print(f"\nGlobal Model Overall Metrics:")
print(f"  MAE:   {overall['MAE']:>12,.2f}")
print(f"  RMSE:  {overall['RMSE']:>12,.2f}")
print(f"  MAPE:  {overall['MAPE']:>10.2f}%")
print(f"  SMAPE: {overall['SMAPE']:>10.2f}%")

# Feature importance
fi_df = pd.DataFrame({
    'feature': best_model.models_['LGBMRegressor'].feature_name_,
    'importance': best_model.models_['LGBMRegressor'].feature_importances_
})
fi_df['pct'] = fi_df['importance'] / fi_df['importance'].sum() * 100
fi_df = fi_df.sort_values('importance', ascending=False).reset_index(drop=True)
print(f"\nTop 10 Features:")
print(fi_df.head(10)[['feature', 'pct']].to_string(index=False))

## 12. Per-Cluster Model Training

Train separate LightGBM models for each profile cluster to compare against the global model.

In [ ]:
# Per-cluster model training using best transform
cluster_names_sorted = sorted(df['profile_cluster'].unique())
results_cluster = {}

print(f"Training per-cluster models with transform: {best_transform}")
print(f"Clusters: {cluster_names_sorted}")
print("=" * 70)

for cluster_name in cluster_names_sorted:
    cluster_df = df[df['profile_cluster'] == cluster_name].copy()
    n_series = cluster_df[unique_id].nunique()
    n_rows = len(cluster_df)
    
    print(f"\n{'â”€'*50}")
    print(f"Cluster: {cluster_name} ({n_series} series, {n_rows} rows)")
    print(f"{'â”€'*50}")
    
    # Check if enough data for lag26
    min_rows = cluster_df.groupby(unique_id).size().min()
    if min_rows < max(lags) + 1:
        usable_lags = [l for l in lags if l < min_rows]
        if not usable_lags:
            print(f"  âš ï¸ Skipping â€” insufficient data (min {min_rows} rows)")
            continue
        print(f"  âš ï¸ Reduced lags to {usable_lags} (min rows: {min_rows})")
    else:
        usable_lags = lags
    
    cluster_static = [c for c in static_features_list if c in cluster_df.columns]
    cluster_dynamic = [c for c in dynamic_cols if c in cluster_df.columns]
    
    t0 = time.time()
    preds, model = rolling_train_and_predict_with_transform_oos_v2(
        df=cluster_df,
        lags=usable_lags,
        freq=freq,
        id_col=unique_id,
        y_col=y,
        static_features=cluster_static,
        dynamic_features=cluster_dynamic,
        eval_date_training=eval_dates_rolling,
        eval_date_oos=eval_date_oos,
        horizon=horizon,
        target_transforms=get_target_transforms_from_name(best_transform),
    )
    elapsed = time.time() - t0
    
    valid_preds = preds.dropna(subset=[y])
    if not valid_preds.empty:
        mae = np.mean(np.abs(valid_preds[y].values - valid_preds['LGBMRegressor'].values))
        print(f"  âœ… {elapsed:.1f}s | {len(valid_preds)} valid preds | MAE: {mae:,.2f}")
    else:
        mae = np.nan
        print(f"  âœ… {elapsed:.1f}s | {len(preds)} preds (no actuals)")
    
    results_cluster[cluster_name] = (preds, model, mae)

print("\nâœ… All per-cluster models complete")

## 13. Global vs Per-Cluster Comparison

In [ ]:
# Compare global vs per-cluster models
print("=" * 70)
print("GLOBAL vs PER-CLUSTER COMPARISON")
print("=" * 70)
print(f"\n{'Model':<20} {'Series':<8} {'MAE':>12} {'vs Global':>12}")
print("â”€" * 55)

global_mae = scores[best_transform]
print(f"{'Global (std)':<20} {'50':<8} {global_mae:>12,.2f} {'baseline':>12}")

# Weighted average of cluster MAEs
weighted_sum = 0
total_series = 0
for cname in sorted(results_cluster.keys()):
    _, _, cluster_mae = results_cluster[cname]
    n = df[df['profile_cluster'] == cname][unique_id].nunique()
    pct_diff = (cluster_mae - global_mae) / global_mae * 100
    sign = "+" if pct_diff > 0 else ""
    print(f"  {cname:<18} {n:<8} {cluster_mae:>12,.2f} {sign}{pct_diff:>10.1f}%")
    weighted_sum += cluster_mae * n
    total_series += n

weighted_avg_mae = weighted_sum / total_series
pct_diff = (weighted_avg_mae - global_mae) / global_mae * 100
sign = "+" if pct_diff > 0 else ""
print("â”€" * 55)
print(f"{'Cluster weighted avg':<20} {total_series:<8} {weighted_avg_mae:>12,.2f} {sign}{pct_diff:>10.1f}%")

# Decision
if weighted_avg_mae < global_mae:
    recommended = "per-cluster"
    print(f"\nâœ… RECOMMENDATION: Per-cluster models ({weighted_avg_mae:,.0f}) beat global ({global_mae:,.0f})")
else:
    recommended = "global"
    print(f"\nâœ… RECOMMENDATION: Global model ({global_mae:,.0f}) beats per-cluster weighted avg ({weighted_avg_mae:,.0f})")

## 14. Save Forecasts to Lakehouse

In [ ]:
# Save OOS forecasts to lakehouse
forecast_table_name = f"{scenario_name}_forecasts"

# Global OOS predictions (last cutoff = last training date)
best_oos_preds = best_preds[best_preds['cutoff_date'] == best_preds['cutoff_date'].max()].copy()
forecast_output = best_oos_preds[[unique_id, 'ds', 'LGBMRegressor']].copy()
forecast_output = forecast_output.rename(columns={'ds': 'FORECAST_DATE', 'LGBMRegressor': 'FORECAST_VALUE'})
forecast_output['MODEL_TYPE'] = 'global'
forecast_output['TRANSFORM'] = best_transform
forecast_output['GENERATED_DATE'] = pd.Timestamp.now()

# Per-cluster OOS predictions
cluster_forecasts = []
for cname, (c_preds, c_model, c_mae) in results_cluster.items():
    c_oos = c_preds[c_preds['cutoff_date'] == c_preds['cutoff_date'].max()].copy()
    c_out = c_oos[[unique_id, 'ds', 'LGBMRegressor']].copy()
    c_out = c_out.rename(columns={'ds': 'FORECAST_DATE', 'LGBMRegressor': 'FORECAST_VALUE'})
    c_out['MODEL_TYPE'] = f'cluster_{cname}'
    c_out['TRANSFORM'] = best_transform
    c_out['GENERATED_DATE'] = pd.Timestamp.now()
    cluster_forecasts.append(c_out)

all_forecasts = pd.concat([forecast_output] + cluster_forecasts, ignore_index=True)
print(f"Total forecast rows: {len(all_forecasts)} (Global: {len(forecast_output)}, Per-cluster: {sum(len(cf) for cf in cluster_forecasts)})")

# Save to lakehouse
spark_forecast = spark.createDataFrame(all_forecasts)
spark_forecast.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(forecast_table_name)
print(f"\nâœ… Saved to lakehouse table: {forecast_table_name}")

## 15. Summary

### Results

| Model | MAE | MAPE | SMAPE |
|-------|-----|------|-------|
| Global (std) | 27,991 | 6.32% | 6.44% |
| Global (identity) | 33,579 | â€” | â€” |
| Global (diff1) | 110,530 | â€” | â€” |
| Per-cluster weighted avg | 27,080 | â€” | â€” |

### Per-Cluster Results

| Cluster | Stores | MAE | vs Global |
|---------|--------|-----|----------|
| erratic | 12 | 25,751 | -8.0% |
| regular_0 | 5 | 26,624 | -4.9% |
| regular_1 | 15 | 29,357 | +4.9% |
| regular_2 | 4 | 30,501 | +9.0% |
| regular_3 | 14 | 24,964 | -10.8% |

### Recommendation

**Per-cluster models** provide a 3.3% improvement over the global model (MAE 27,080 vs 27,991).  
Largest gains from regular_3 (-10.8%) and erratic (-8.0%) clusters.

### Top Features (Global Model)

| Rank | Feature | Importance % |
|------|---------|-------------|
| 1 | SERVICE_GAP_PRODUCE | 3.83% |
| 2 | lag4 | 3.77% |
| 3 | DISCOUNT_MEMBER_PRICING | 3.53% |
| 4 | lag13 | 3.47% |
| 5 | AVG_SERVICE_GAP_RATIO | 3.47% |
| 6 | UNIQUE_LOYALTY_CARDS | 3.30% |
| 7 | lag8 | 3.13% |
| 8 | SERVICE_GAP_DELI | 2.97% |
| 9 | lag26 | 2.83% |
| 10 | AVG_WEEKLY_TEMPERATURE | 2.63% |